# LlamaFactory LoRA 全流程实战向导：微调 · 推理 · 测试集预测 · 评估 · 权重合并 · vLLM 部署

本 Notebook 以**命令行示意**为主，完整覆盖从数据准备、LoRA 微调训练、模型推理对话、测试集预测、性能评估、权重合并导出到vLLM 高性能推理部署的端到端全流程，适合作为 LlamaFactory 上手实践的一站式参考向导。

---
**硬件需求**：显存 ≥ 16GB（7B 模型 + LoRA + bf16），推荐 A100 / 3090 / 4090  
**框架版本**：LLaMA Factory ≥ 0.9.0 · vLLM ≥ 0.4.0

## 一、环境安装

### 1.1 安装 LLaMA Factory

克隆官方仓库并以可编辑模式安装，`[torch,metrics]` 会同时安装训练和评估所需的依赖：

```bash
git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
cd LLaMA-Factory
pip install -e ".[torch,metrics]"
```

### 1.2 验证安装

```bash
llamafactory-cli version
nvidia-smi
```

## 二、数据集准备

### 2.1 数据格式

LLaMA Factory 支持两种主流数据格式，按数据来源和对话轮次选择：

---

#### 格式一：Alpaca 格式（单轮，推荐用于指令微调）

适合**单轮问答**场景，每条样本由 instruction + input + output 三部分构成。  
保存到 `data/my_alpaca.json`：

```json
[
  {
    "instruction": "请将以下文本翻译成英文",
    "input": "人工智能正在改变世界。",
    "output": "Artificial intelligence is changing the world."
  },
  {
    "instruction": "写一首关于春天的五言绝句",
    "input": "",
    "output": "春风吹绿岸，燕子衔泥忙。花开千树艳，鸟鸣一声长。"
  }
]
```

| 字段 | 是否必填 | 说明 |
|---|---|---|
| `instruction` | 必填 | 任务指令，最终拼接为 prompt |
| `input` | 可为空字符串 | 用户的具体输入内容，为空时只用 instruction |
| `output` | 必填 | 期望的模型回答 |
| `system` | 可选 | 为该条样本单独指定 system prompt |

---

#### 格式二：ShareGPT 格式（多轮，推荐用于对话微调）

适合**多轮对话**场景，每条样本包含一段完整的对话历史。  
保存到 `data/my_sharegpt.json`：

```json
[
  {
    "system": "你是一个有帮助的中文AI助手。",
    "conversations": [
      {
        "from": "human",
        "value": "你好，请介绍一下大语言模型。"
      },
      {
        "from": "gpt",
        "value": "大语言模型（LLM）是基于 Transformer 架构、在海量文本上预训练的神经网络。"
      },
      {
        "from": "human",
        "value": "它和普通的神经网络有什么区别？"
      },
      {
        "from": "gpt",
        "value": "主要区别在于规模和预训练范式：LLM 参数量通常在 10亿以上，且通过自监督方式在互联网级语料上训练。"
      }
    ]
  }
]
```

| 字段 | 是否必填 | 说明 |
|---|---|---|
| `system` | 可选 | 全局 system prompt，适用于该条对话的所有轮次 |
| `conversations` | 必填 | 对话轮次列表，按时间顺序排列 |
| `conversations[i].from` | 必填 | 发言角色，`human` 表示用户，`gpt` 表示模型 |
| `conversations[i].value` | 必填 | 该轮次的具体文本内容 |

> **如何选择格式：**  
> - 数据来自人工标注的单轮问答 → Alpaca 格式  
> - 数据来自 ChatGPT / Claude 的多轮对话导出 → ShareGPT 格式  
> - 两种格式可以在 `dataset` 字段中混用，框架自动识别

### 2.2 注册数据集

编辑 `data/dataset_info.json`，两种格式的注册方式略有不同：

**Alpaca 格式注册**（通过 `columns` 映射字段名）：

```json
{
  "my_alpaca": {
    "file_name": "my_alpaca.json",
    "columns": {
      "prompt":   "instruction",
      "query":    "input",
      "response": "output",
      "system":   "system"
    }
  }
}
```

**ShareGPT 格式注册**（需额外声明 `formatting` 和 `tags` 定义角色标识）：

```json
{
  "my_sharegpt": {
    "file_name": "my_sharegpt.json",
    "formatting": "sharegpt",
    "columns": {
      "messages": "conversations",
      "system":   "system"
    },
    "tags": {
      "role_tag":      "from",
      "content_tag":   "value",
      "user_tag":      "human",
      "assistant_tag": "gpt"
    }
  }
}
```

| 字段 | 说明 |
|---|---|
| `formatting` | Alpaca 格式省略此字段；ShareGPT 格式必须填 `"sharegpt"` |
| `columns.messages` | JSON 中对话列表对应的字段名（本例为 `conversations`） |
| `tags.role_tag` | 每条消息中标识角色的键名（本例为 `from`） |
| `tags.user_tag` | 用户角色的值（本例为 `human`） |
| `tags.assistant_tag` | 模型角色的值（本例为 `gpt`） |

---

**`file_name` 路径说明**

| 写法 | 起点 | 示例 | 适用场景 |
|---|---|---|---|
| 相对路径 | `dataset_info.json` 所在目录（默认 `data/`） | `"my_alpaca.json"` → `data/my_alpaca.json` | 数据集在 `data/` 目录下（推荐） |
| `..` 相对路径 | 同上，通过 `..` 向上跳出 `data/` | `"../raw/train.json"` → 项目根下的 `raw/train.json` | 可用，但路径深时难以维护 |
| **绝对路径** | **文件系统根目录，与 `data/` 完全无关** | `"/home/user/datasets/train.json"` | 数据集在 `data/` 之外时推荐使用 |

> 绝对路径**不需要经过 `data/` 目录**，直接从磁盘根（Linux: `/`，Windows: `C:\`）出发定位文件，框架检测到绝对路径后会跳过拼接直接加载。

---

注册后在训练配置的 `dataset` 字段中填入**键名**（即 `dataset_info.json` 中的 JSON key，不是文件名），**多个数据集可以用逗号拼接混用**：

```yaml
# 填写的是 dataset_info.json 中的键名（my_alpaca / my_sharegpt），而非文件名（my_alpaca.json）
dataset: my_alpaca,my_sharegpt   # 框架根据键名查找注册信息，自动合并并打乱顺序
```

### 2.3 思维链（CoT）数据集支持

DeepSeek-R1、Qwen3 等推理模型在回答前会先输出 `<think>...</think>` 包裹的推理过程，再给出最终答案。

LLaMA Factory 对思维链数据的处理方式：**将 `<think>...</think>` 和最终答案拼接后，整体写入 `output`（Alpaca）或 `gpt` 角色的 `value`（ShareGPT）字段**，框架将其作为普通目标序列进行 SFT 训练，无需任何额外配置。

---

#### Alpaca 格式

```json
[
  {
    "instruction": "计算 15 的阶乘",
    "input": "",
    "output": "<think>\n15! = 15 × 14 × ... × 1，逐步累乘得到结果。\n</think>\n\n15! = 1307674368000"
  },
  {
    "instruction": "判断 127 是否为质数",
    "input": "",
    "output": "<think>\n检验 2 到 √127 ≈ 11.3 之间的整数：2、3、5、7、11 均不能整除 127。\n</think>\n\n127 是质数。"
  }
]
```

`dataset_info.json` 注册与普通 Alpaca 数据集完全相同，**无需新增任何字段**：

```json
{
  "my_cot_alpaca": {
    "file_name": "my_cot_alpaca.json",
    "columns": {
      "prompt":   "instruction",
      "query":    "input",
      "response": "output"
    }
  }
}
```

---

#### ShareGPT 格式（多轮对话）

`<think>...</think>` 写在 `gpt` 角色的 `value` 中，与最终回答拼接在一起：

```json
[
  {
    "conversations": [
      {"from": "human", "value": "判断 127 是否为质数"},
      {"from": "gpt",   "value": "<think>\n检验 2 到 √127 ≈ 11.3 之间的整数均不能整除 127。\n</think>\n\n127 是质数。"}
    ]
  }
]
```

`dataset_info.json` 注册同样无需额外字段：

```json
{
  "my_cot_sharegpt": {
    "file_name": "my_cot_sharegpt.json",
    "formatting": "sharegpt",
    "columns": {
      "messages": "conversations"
    }
  }
}
```

---

#### `enable_thinking` 参数（推理型模型专用训练配置）

对于具备推理能力的模型（如 Qwen3），LLaMA Factory 提供 `enable_thinking` 参数控制训练时思维链的处理方式。

| `enable_thinking` | 模式 | 输出中**已有** `<think>...</think>` | 输出中**无** `<think>...</think>` |
|---|---|---|---|
| `true`（默认） | 慢思考 | 检测到已有思维链 → **保留原样，不额外插入** | 自动在 `output` 前插入空 `<think></think>`，**计入 loss** |
| `false` | 快思考 | **先剥除已有思维链内容**，再向 prompt 末尾追加空 `<think></think>`，**不计入 loss** | 直接向 prompt 末尾追加空 `<think></think>`，**不计入 loss** |
| `null` | 混合模式 | 含 CoT 的样本走慢思考，不含 CoT 的走快思考（配置较复杂，谨慎使用） | 同左 |

> ⚠️ **Alpaca 格式数据含 CoT 时的注意事项**：Alpaca 格式中思维链只能放在 `output` 字段（如 `<think>推理过程</think>最终答案`）。此时若设置 `enable_thinking: false`，框架会**先将 `output` 中的 `<think>...</think>` 全部删除**，再在 prompt 侧追加空标签，导致真实推理链内容丢失，训练效果受损。因此：
> - 数据含真实 CoT → 应使用 `enable_thinking: true`（默认）或不设置该参数
> - 仅当数据**不含 CoT** 且想训练快思考模式时，才应设置 `enable_thinking: false`

对于同时提供推理版和非推理版的模型（如 Qwen3 系列），若只需训练**非推理模式**，使用 `_nothink` 后缀模板，可完全去掉 `<think>` token：

```yaml
# 方式一：完全不包含 <think> token
template: qwen3_nothink

# 方式二：使用官方 fast thinking 格式（与官方 enable_thinking=False 推理行为一致）
template: qwen3
enable_thinking: false
```

> **两者的区别**：`qwen3_nothink` 直接不生成 `<think>` 相关内容；`qwen3` + `enable_thinking: false` 会在 prompt 侧插入空 `<think></think>` 但不计入 loss，其训练时的 prompt 构造方式与 Qwen3 官方 Transformers API（`apply_chat_template(enable_thinking=False)`）严格对齐，迁移性更好。
>
> ⚠️ **注意**：上述配置仅控制**训练阶段**的数据处理行为。使用 LlamaFactory 进行推理时（`llamafactory-cli chat` 或 `do_predict: true`），**无论训练时如何设置，推理配置文件中都需要手动加上 `enable_thinking: false`** 才能关闭思维链输出；若不设置，默认仍会生成 `<think>...</think>` 推理过程。

---

> ⚠️ **对所有带推理链（CoT）能力的模型的特别说明**
>
> `enable_thinking` 参数并非 Qwen3 专属，LLaMA Factory 会对所有模型统一应用该逻辑。当使用默认值 `enable_thinking: true` 且训练数据**不含 CoT**（即无 `<think>...</think>` 内容）时，框架会自动在每条样本的 `output` 前拼接空的 `<think></think>` 并**计入 loss**，导致模型学习到"直接输出答案、不进行推理"的行为模式。
>
> **实测结果**：以 DeepSeekR1DistillLlama8B 为例，使用无 CoT 数据 + `enable_thinking: true`（默认）微调后，推理时输出中**不再出现 `<think>` 标签**，模型原有的推理链能力遭到**不可逆的破坏与丧失**——这本质上是一种**灾难性遗忘**：模型通过大量梯度更新反复学习"空 `<think></think>` → 直接输出答案"的行为模式，逐步覆盖并损毁了原有的推理权重，而非仅仅在推理阶段"暂时抑制"了该能力。该结论同样适用于其他具备推理能力的模型，例如：
> - **DeepSeek-R1 系列**：DeepSeek-R1、DeepSeekR1-Distill-Qwen/Llama 等蒸馏变体
> - **QwQ 系列**：QwQ-32B 等阿里巴巴推出的推理模型
> - **Marco-o1**：AIDC-AI 基于 Qwen2.5 蒸馏的开源推理模型
> - **Skywork-o1**：Skywork 团队基于 Llama 系列蒸馏的推理模型
>
> **建议**：
> - 若希望**保留**模型的推理能力，应在训练数据中包含 `<think>...</think>` 格式的 CoT 内容；或在数据集**不含 CoT** 时，显式设置 `enable_thinking: false`，避免框架将空 `<think></think>` 计入 loss 而破坏推理权重。

### 2.4 多模态数据集支持

LLaMA Factory ≥ 0.9.0 支持**图像、视频、音频**三种模态的数据集，两种数据格式（Alpaca / ShareGPT）均可使用。多模态数据集需要搭配对应的**视觉语言模型**（VLM）使用，如 Qwen2-VL、InternVL、LLaVA、Llama 4 等。

---

#### 核心机制：占位符（Placeholder）

在文本字段中嵌入占位符，告知框架在该位置插入对应的媒体内容：

| 模态 | 占位符 | 数据字段 |
|---|---|---|
| 图像 | `<image>` | `images`（路径列表） |
| 视频 | `<video>` | `videos`（路径列表） |
| 音频 | `<audio>` | `audios`（路径列表） |

> **严格匹配规则**：文本中占位符的**数量**必须与媒体路径列表的**长度**完全一致，否则框架报错。

---

#### 图像数据集

**Alpaca 格式**（单轮，图像路径挂在顶层 `images` 字段，`instruction` 中用 `<image>` 标记插入位置）：

```json
[
  {
    "instruction": "<image>请描述这张图片的内容。",
    "input": "",
    "output": "图中是一片金色的麦田，远处有蓝天白云，画面宁静开阔。",
    "images": ["data/images/wheat_field.jpg"]
  },
  {
    "instruction": "比较这两张图片的风格差异：<image> 和 <image>",
    "input": "",
    "output": "左图为水墨画风格，笔触简练；右图为油画风格，色彩浓烈。",
    "images": ["data/images/ink_painting.jpg", "data/images/oil_painting.jpg"]
  }
]
```

在 `dataset_info.json` 中注册（增加 `columns.images` 映射）：

```json
{
  "my_image_alpaca": {
    "file_name": "my_image_alpaca.json",
    "columns": {
      "prompt":   "instruction",
      "query":    "input",
      "response": "output",
      "images":   "images"
    }
  }
}
```

---

**ShareGPT 格式**（多轮对话，`<image>` 占位符写在 `conversations` 的 `value` 中）：

```json
[
  {
    "conversations": [
      {"from": "human", "value": "<image>这张图里有什么动物？"},
      {"from": "gpt",   "value": "图中有一只金毛寻回犬，正在草地上玩耍。"},
      {"from": "human", "value": "它大概几岁？"},
      {"from": "gpt",   "value": "从毛色和体型判断，大约是 1～2 岁的幼犬。"}
    ],
    "images": ["data/images/golden_retriever.jpg"]
  }
]
```

在 `dataset_info.json` 中注册（增加 `columns.images` 映射）：

```json
{
  "my_image_sharegpt": {
    "file_name": "my_image_sharegpt.json",
    "formatting": "sharegpt",
    "columns": {
      "messages": "conversations",
      "images":   "images"
    },
    "tags": {
      "role_tag":      "from",
      "content_tag":   "value",
      "user_tag":      "human",
      "assistant_tag": "gpt"
    }
  }
}
```

---

#### 视频数据集

结构与图像完全一致，将字段名和占位符替换为 `videos` / `<video>`：

```json
[
  {
    "instruction": "<video>请总结这段视频的主要内容。",
    "input": "",
    "output": "视频展示了一段延时摄影，记录了花朵从含苞到盛开的完整过程，历时约 24 小时。",
    "videos": ["data/videos/flower_timelapse.mp4"]
  }
]
```

`dataset_info.json` 注册：

```json
{
  "my_video_dataset": {
    "file_name": "my_video_dataset.json",
    "columns": {
      "prompt":   "instruction",
      "query":    "input",
      "response": "output",
      "videos":   "videos"
    }
  }
}
```

---

#### 音频数据集

同理，将字段名和占位符替换为 `audios` / `<audio>`：

```json
[
  {
    "instruction": "<audio>请转录这段音频，并判断说话人的情绪。",
    "input": "",
    "output": "转录文本：「今天真的太开心了，终于拿到 offer！」\n情绪判断：兴奋、喜悦。",
    "audios": ["data/audios/speech_sample.wav"]
  }
]
```

`dataset_info.json` 注册：

```json
{
  "my_audio_dataset": {
    "file_name": "my_audio_dataset.json",
    "columns": {
      "prompt":   "instruction",
      "query":    "input",
      "response": "output",
      "audios":   "audios"
    }
  }
}
```

---

#### 混合模态（图像 + 音频）

一条样本可以同时包含多种模态，各字段独立列举，占位符在文本中分别标记位置：

```json
[
  {
    "instruction": "根据图片 <image> 和对应的语音描述 <audio>，判断两者内容是否一致。",
    "input": "",
    "output": "一致。图片展示的是一只猫趴在键盘上，语音描述的内容与之相符。",
    "images": ["data/images/cat_keyboard.jpg"],
    "audios": ["data/audios/cat_description.wav"]
  }
]
```

`dataset_info.json` 注册（同时声明 `images` 和 `audios`）：

```json
{
  "my_mixed_dataset": {
    "file_name": "my_mixed_dataset.json",
    "columns": {
      "prompt":   "instruction",
      "query":    "input",
      "response": "output",
      "images":   "images",
      "audios":   "audios"
    }
  }
}
```

---

#### 媒体文件路径说明

| 路径写法 | 说明 |
|---|---|
| 相对路径（如 `"images/a.jpg"`） | 框架将其拼接到训练配置中 `media_dir` 的值之后；若未设置 `media_dir`，则相对于**命令执行时的工作目录**（不是 `dataset_info.json` 所在目录） |
| 绝对路径（如 `"/home/user/images/a.jpg"`） | 直接定位文件，与 `media_dir` 完全无关，跨服务器迁移时需注意路径一致性 |
| HuggingFace Hub 路径 | 数据集托管在 Hub 上时，框架自动下载媒体文件 |

> **`media_dir` 字段**：在训练 YAML 配置中指定 `media_dir: /path/to/media/root`，框架会自动将数据文件中的所有相对路径拼接到该根目录之后。**推荐做法**：将所有媒体文件集中放到一个目录，`media_dir` 指向该目录，数据 JSON 中只写文件名（如 `"a.jpg"`），迁移时只需改 `media_dir` 一处即可。

---

#### 支持的视觉语言模型（VLM）

多模态数据集必须搭配支持对应模态的模型，训练时 `template` 字段须填写对应模板：

| 模型系列 | 支持模态 | `template` |
|---|---|---|
| Qwen2-VL / Qwen2.5-VL / QVQ | 图像、视频 | `qwen2_vl` |
| Qwen2.5-Omni | 图像、视频、音频 | `qwen2_omni` |
| InternVL 2.5 / 3.x | 图像、视频 | `intern_vl` |
| LLaVA-1.5 | 图像 | `llava` |
| LLaVA-NeXT | 图像 | `llava_next` |
| Llama 4 | 图像 | `llama4` |
| Mllama（Llama 3.2 Vision） | 图像 | `mllama` |
| Gemma3 | 图像 | `gemma3` |

> **注意**：纯文本模型（如 LLaMA-3、Qwen2.5 等）不支持多模态数据集，若强行传入 `images` 字段会被框架忽略或报错。

---

#### 注意事项

1. **占位符数量严格匹配**：文本中有几个 `<image>`，`images` 列表就必须有几个路径，多或少都会报错。
2. **媒体文件格式**：图像支持 JPEG/PNG/WebP 等常见格式；视频支持 MP4/AVI 等；音频支持 WAV/MP3/FLAC 等，具体取决于所用模型的处理器（Processor）。
3. **显存需求大幅增加**：图像/视频会被编码为大量 token（如一张 1024×1024 图像约占 256～1024 个视觉 token），相比纯文本微调显存占用显著上升，建议开启 `gradient_checkpointing: true`。
4. **`cutoff_len` 需适当增大**：视觉 token 会占用序列长度配额，建议将 `cutoff_len` 设为 4096 及以上，避免大量样本被截断。

## 三、LoRA 微调训练

### 3.1 修改训练配置文件

训练配置文件位于 `configs/lora_LlamaFactory_train.yaml`，启动训练前修改以下**必填字段**：

| 字段 | 说明 |
|---|---|
| `model_name_or_path` | 基础模型本地路径或 HuggingFace Hub ID |
| `template` | 对话模板，**必须与基础模型自身的对话模板完全一致**（如 `qwen` / `llama3` / `glm4`），完整对照见 §4.1 注释 |
| `dataset` | 在 `dataset_info.json` 中注册的数据集名称 |
| `output_dir` | 训练结果（LoRA 权重 + 检查点）的保存路径 |

其余参数（LoRA rank/alpha、学习率、早停等）均已配有详细注释，按需调整。

> ⚠️ **`template` 字段必须与基础模型的对话模板严格一致**
>
> `template` 决定了框架如何将数据格式化为模型输入（包括 system prompt 包裹方式、角色标记、特殊 token 等）。该格式由基础模型在预训练 / 指令微调阶段固定下来，写在模型的 `tokenizer_config.json` 的 `chat_template` 字段中。
>
> **若 `template` 填错，会导致：**
> - 训练时数据被格式化成模型从未见过的格式，微调无效甚至负向迁移；
> - 推理时 system / user / assistant 标记混乱，模型输出质量大幅下降；
> - 常见错误：Qwen 系列填成 `llama3`，或将 Qwen3 系列填成旧版 `qwen`。
>
> **确认方法**：查看模型 `tokenizer_config.json` 中的 `chat_template` 字段，或参照下表（完整列表见 §4.1 注释）。

| 基础模型系列 | 正确 `template` 值 |
| --- | --- |
| Qwen / Qwen1.5 / Qwen2 / Qwen2.5 | `qwen` |
| Qwen3 / Qwen3.5（思考模式） | `qwen3` / `qwen3_5` |
| Qwen3 / Qwen3.5（非思考模式） | `qwen3_nothink` / `qwen3_5_nothink` |
| LLaMA-3 / LLaMA-3.3 | `llama3` |
| GLM-4 / GLM-Z1 | `glm4` / `glmz1` |
| InternLM-2 / InternLM-3 | `intern2` |
| Mistral / Mixtral | `mistral` |
| DeepSeek（LLM / Code / MoE） | `deepseek` |
| DeepSeek-R1（Distill 系列） | `deepseekr1` |

### 3.2 启动训练

**单卡训练**

```bash
llamafactory-cli train configs/lora_LlamaFactory_train.yaml
```

**多卡分布式训练（DeepSpeed）**

```bash
# --num_gpus 指定 GPU 卡数；deepspeed 字段在 YAML 中指定 Zero 策略配置文件
deepspeed --num_gpus=4 \
    $(which llamafactory-cli) train configs/lora_LlamaFactory_train.yaml
```

在训练配置文件中取消注释 `deepspeed` 字段以启用对应的 Zero 优化策略：

```yaml
# Zero-2：优化器状态 + 梯度跨卡分片，模型参数每卡完整保留（适合大多数场景）
deepspeed: examples/deepspeed/ds_z2_config.json

# Zero-3：在 Zero-2 基础上进一步将模型参数也跨卡分片（适合超大模型显存不足时）
# deepspeed: examples/deepspeed/ds_z3_config.json
```

> `examples/deepspeed/` 目录为 LLaMA Factory 仓库内置的配置文件，克隆仓库后即可直接使用。

**训练完成后，`output_dir` 目录结构如下：**

```
output/lora_sft/
├── checkpoint-500/                  # 中间检查点
├── adapter_model.safetensors        # 最终 LoRA 权重（仅几十 MB）
├── adapter_config.json              # LoRA 结构配置
├── trainer_state.json               # 训练历史与最优检查点信息
└── training_loss.png                # 训练 / 验证损失曲线图
```

> **判断训练质量：** train_loss 和 eval_loss 同步下降 → 正常；eval_loss 反弹而 train_loss 继续下降 → 过拟合，减少 epoch 或增大 lora_dropout。

## 四、推理（Chat 模式）

### 4.1 准备推理配置文件

新建 `configs/lora_LlamaFactory_infer.yaml`，填入以下关键字段：

```yaml
model_name_or_path: /path/to/your/base/model   # 基础模型路径
adapter_name_or_path: ./output/lora_sft        # 训练输出的 LoRA 权重目录
finetuning_type: lora                          # 告知框架加载 LoRA 适配器
# template 须与训练时以及基础模型自身的对话模板保持完全一致（三者必须统一）：
#   基础模型在预训练阶段使用的对话模板已固化在 tokenizer_config.json 的
#   chat_template 字段中，训练和推理时均须填写与之对应的 LLaMA Factory 模板名。
#   常见模板名对应关系（完整列表见官方 README）：
#   Qwen / Qwen1.5 / Qwen2 / Qwen2.5          → qwen
#   Qwen3 / Qwen3.5（思考模式）                → qwen3 / qwen3_5
#   Qwen3 / Qwen3.5（非思考模式）              → qwen3_nothink / qwen3_5_nothink
#   LLaMA 3 / LLaMA 3.3                       → llama3
#   GLM-4 / GLM-Z1                            → glm4 / glmz1
#   InternLM 2 / InternLM 3                   → intern2
#   Mistral / Mixtral                         → mistral
#   DeepSeek (LLM/Code/MoE)                  → deepseek
#   DeepSeek-R1 (Distill)                     → deepseekr1
template: qwen                                 # 示例：Qwen / Qwen2 / Qwen2.5 系列

# 生成参数
max_new_tokens: 512
temperature: 0.7
top_p: 0.9
```

框架会自动在内存中将 LoRA 适配器合并到基础模型再进行推理，**无需提前合并权重**。

---

#### `enable_thinking` 在推理配置中的用法（推理型模型）

`enable_thinking` **同样可以写在推理/预测的 YAML 配置中**，用于控制推理阶段是否输出思维链。不同模型系列的支持情况不同：

| 模型系列 | 有 `_nothink` 模板 | 可用 `enable_thinking` |
|---|---|---|
| Qwen3 / Qwen3.5 / Qwen3-VL / Qwen3-Omni | ✅ 有（推荐） | ✅ 可用 |
| DeepSeek-R1 / DeepSeek-R1-Distill 系列 | ❌ 无 | ✅ **唯一方式** |
| QwQ / Marco-o1 / Skywork-o1 等其他推理模型 | ❌ 无 | ✅ 可用 |

**两种关闭思考链的方式对比：**

| 控制方式 | 用法示例 | 说明 |
|---|---|---|
| 模板后缀（Qwen3 专属） | `template: qwen3_nothink` | 语义清晰，官方示例首选；**仅 Qwen 系列有此模板** |
| `enable_thinking: false` | 在 YAML 中添加该字段 | 通用方式，**DeepSeek-R1 等无 `_nothink` 模板的模型只能用此方式**；用于预测阶段（`do_predict: true`）时需版本 ≥ 0.9.4 |

> `enable_thinking` 仅对具备推理能力的模型有实际效果，普通模型（如 Llama3-Instruct 原版）设置后无影响。

### 4.2 启动交互式对话

```bash
llamafactory-cli chat configs/lora_LlamaFactory_infer.yaml
```

运行后在终端直接输入问题，框架返回模型回答，输入 `exit` 或按 `Ctrl+C` 退出。

**也可以启动 Web UI 进行可视化对话：**

```bash
llamafactory-cli webui
# 打开浏览器访问 http://localhost:7860
# 在 Chat 标签页填入模型路径和 LoRA 路径后即可对话
```

## 五、基准评估（Eval 模式）

### 5.1 评估原理与支持的基准

`llamafactory-cli eval` 专门用于在**多选题形式的标准基准**上评测模型能力，与训练时的 `eval_loss` 不同：

| 对比维度 | 训练验证（`eval_loss`） | 基准评估（`llamafactory-cli eval`） |
|---|---|---|
| 触发时机 | 训练过程中周期性执行 | 训练结束后独立运行 |
| 评测方式 | 对 ground-truth token 计算交叉熵损失 | 让模型从 ABCD 四选项中选出正确答案，计算准确率（Accuracy） |
| 输出指标 | `eval_loss`（越低越好） | 各学科准确率 + 整体平均准确率（越高越好） |
| 适用场景 | 监控收敛、判断过拟合 | 横向对比、与论文结果比较、上报排行榜 |

---

**官方支持的评估基准（`task` 字段的合法值仅以下三个）**

| `task` 字段值 | 基准名称 | 语言 | 学科数 | 题目总数 | 标准 `n_shot` | 标准 `lang` |
|---|---|---|---|---|---|---|
| `ceval_validation` | C-Eval（验证集） | 中文 | 52 | 13,948 | 5 | `zh` |
| `cmmlu_test` | CMMLU（测试集） | 中文 | 67 | 11,528 | 5 | `zh` |
| `mmlu_test` | MMLU（测试集） | 英文 | 57 | 14,042 | 5 | `en` |

> **注意**：`task` 值的格式为 `{基准名}_{split}`，分割信息（`validation`/`test`）已编码在任务名中，配置文件中**没有单独的 `split` 字段**。`arc_challenge`、`hellaswag` 等其他基准名不是合法的 `task` 值，填写后框架会报错。

---

> **数据集下载方式（以 C-Eval 为例，存放到默认目录 `evaluation/`）**：
> ```bash
> # 方式一：HuggingFace CLI（需配置镜像或科学上网）
> # C-Eval 官方 Hub ID 为 ceval/ceval-exam
> huggingface-cli download ceval/ceval-exam --local-dir ./evaluation/ceval
>
> # 方式二：ModelScope（国内推荐，无需翻墙）
> pip install modelscope
> python -c "from modelscope import snapshot_download; snapshot_download('ceval/ceval-exam', cache_dir='./evaluation')"
> ```

### 5.2 修改评估配置文件

评估配置文件位于 `configs/lora_LlamaFactory_eval.yaml`，启动评估前修改以下**必填字段**：

| 字段 | 说明 |
|---|---|
| `model_name_or_path` | 基础模型本地路径或 HuggingFace Hub ID |
| `adapter_name_or_path` | 训练输出的 LoRA 权重目录（评估基础模型时注释掉此项和 `finetuning_type`） |
| `template` | **固定填 `fewshot`**，eval 命令专用评估模板，与训练时对话模板无关 |
| `task` | 评估任务，合法值：`ceval_validation` / `cmmlu_test` / `mmlu_test` |
| `task_dir` | 评估数据集的本地存放目录（默认 `evaluation`） |
| `lang` | 提示词语言（`zh` 对应中文基准，`en` 对应英文基准） |
| `save_dir` | 评估结果输出目录（**不能已存在**，否则框架直接报错） |

**评估 LoRA 微调模型 vs 评估基础模型**

```yaml
# ── 评估 LoRA 微调后的模型（保留这两行）────────────────────────
finetuning_type: lora
adapter_name_or_path: ./output/lora_sft

# ── 评估未微调的基础模型（注释掉上面两行）──────────────────────
# finetuning_type: lora
# adapter_name_or_path: ./output/lora_sft
```

> **建议**：分别对基础模型和微调后模型各跑一次评估（`save_dir` 使用不同路径），对比两份 `summary.json` 中的 `average_acc`，可量化验证微调带来的效果提升。

### 5.3 启动评估

```bash
llamafactory-cli eval configs/lora_LlamaFactory_eval.yaml
```

评估完成后，`save_dir` 目录下会生成以下文件：

```
output/eval_results/
├── results.json       # 每个学科的准确率明细（如 ceval 共 52 个学科）
└── summary.json       # 各大类学科准确率汇总 + 整体平均准确率
```

**`summary.json` 示例**（C-Eval 评估结果片段）：

```json
{
  "average_acc": 0.6134,
  "STEM": { "acc": 0.5821 },
  "Social Science": { "acc": 0.6892 },
  "Humanities": { "acc": 0.6453 },
  "Other": { "acc": 0.5971 }
}
```

**微调前后效果对比建议**

```bash
# 第一步：评估基础模型（注释掉 adapter_name_or_path 后运行）
llamafactory-cli eval configs/lora_LlamaFactory_eval.yaml
# 将结果目录改名备份
mv output/eval_results output/eval_results_base

# 第二步：评估 LoRA 微调后模型（恢复 adapter_name_or_path 后运行）
llamafactory-cli eval configs/lora_LlamaFactory_eval.yaml
# 结果保存到 output/eval_results

# 第三步：对比 summary.json 中的 average_acc
# 差值 = 微调后准确率 − 基础模型准确率（正值表示微调有效提升了基准性能）
```

> **注意事项**
> - 评估耗时较长（7B 模型在 C-Eval 上约需 30~60 分钟），建议在训练完成后单独运行
> - 若基准数据集语言与 `lang` 设置不一致，评估结果会偏低（如用英文 prompt 评测中文基准）
> - `n_shot=5` 与论文标准一致，改为 `0` 可测试零样本能力（通常比 5-shot 低 3~8 个百分点）

### 5.4 在自定义数据集上评估（NLG Evaluation）

`llamafactory-cli eval` **不支持自定义数据集**，它只能评测三个内置多选题基准。若要在自己的数据集上做评估，需使用**另一条路径**：`llamafactory-cli train` 配合 `do_predict: true`。

---

**两种评估方式对比**

| 对比维度 | `llamafactory-cli eval` | `llamafactory-cli train` + `do_predict: true` |
|---|---|---|
| 支持数据集 | 仅内置基准（MMLU / C-Eval / CMMLU） | **任意自定义数据集**（在 `dataset_info.json` 中注册即可） |
| 评估命令 | `llamafactory-cli eval` | `llamafactory-cli train`（调用 train 命令，但不做训练） |
| 任务类型 | 多选题（ABCD 选项） | 开放式生成（模型自由生成文本） |
| 输出指标 | 各学科准确率 + 整体平均准确率 | BLEU-4、ROUGE-1、ROUGE-2、ROUGE-L |
| 生成方式 | 对选项计算 log-prob，选概率最高者 | 调用 `model.generate()` 自回归生成完整回答 |
| 关键配置 | `task`、`n_shot`、`lang` | `do_predict: true`、`predict_with_generate: true` |
| 配置文件 | `configs/lora_LlamaFactory_eval.yaml` | `configs/lora_LlamaFactory_predict.yaml` |

---

**启动自定义数据集评估**

```bash
llamafactory-cli train configs/lora_LlamaFactory_predict.yaml
```

运行前需修改 `configs/lora_LlamaFactory_predict.yaml` 中的以下字段：

| 字段 | 说明 |
|---|---|
| `model_name_or_path` | 基础模型路径 |
| `adapter_name_or_path` | LoRA 权重目录（评估基础模型时注释掉） |
| `template` | 对话模板，须与**基础模型自身的对话模板以及训练时**保持完全一致（如 `llama3` / `qwen`），完整对照见 §4.1 注释 |
| `eval_dataset` | 评估数据集键名（需在 `dataset_info.json` 中注册） |
| `dataset_dir` | `dataset_info.json` 所在目录 |
| `output_dir` | 结果输出目录 |
| `max_new_tokens` | 最大生成长度，建议与数据集平均回答长度对齐 |

---

**输出文件结构**

```
output/predict_results/
├── generated_predictions.jsonl   # 每条样本的模型输出（逐行 JSON，可人工审阅）
├── predict_results.json          # BLEU-4 / ROUGE-1 / ROUGE-2 / ROUGE-L 汇总得分
└── all_results.json              # 包含所有指标的完整结果
```

**`predict_results.json` 示例**：

```json
{
  "predict_bleu-4": 28.53,
  "predict_rouge-1": 52.41,
  "predict_rouge-2": 31.87,
  "predict_rouge-l": 48.92,
  "predict_runtime": 312.4,
  "predict_samples_per_second": 3.2
}
```

> **ROUGE-L** 是综合衡量生成质量最常用的指标（基于最长公共子序列 F1），越高越好；**BLEU-4** 侧重精确率，对短文本敏感。两个指标结合判断更全面。

## 六、合并 LoRA 权重

### 6.1 合并原理

LoRA 的低秩矩阵 $B \cdot A$ 乘以缩放系数后直接加回原始权重，得到与原模型结构完全相同的完整模型：

$$W_{merged} = W_{base} + \frac{\alpha}{r} \cdot B \cdot A$$

| | 合并前（LoRA 动态加载） | 合并后（完整模型） |
|---|---|---|
| 磁盘占用 | 基础模型 + 几十 MB | 与基础模型等大 |
| 推理延迟 | 略高（运行时合并） | 与原始模型相同 |
| 适用场景 | 训练调试、快速切换多个 LoRA | 生产部署、进一步量化 |

### 6.2 准备合并配置文件

新建 `configs/lora_LlamaFactory_merge.yaml`：

```yaml
model_name_or_path: /path/to/your/base/model   # 基础模型路径
adapter_name_or_path: ./output/lora_sft        # LoRA 权重目录
finetuning_type: lora
# template 须与基础模型自身的对话模板以及训练时保持完全一致，
# 合并时框架需正确识别特殊 token 才能对齐权重结构（如 llama3 / qwen 等，见 §4.1 注释）
template: llama3

export_dir: ./output/merged_model              # 合并后完整模型的保存路径
export_size: 2                                 # 每个分片文件最大 2 GB
export_device: cuda                            # 合并运算设备（显存不足时改为 cpu）
export_legacy_format: false                    # false=保存为 safetensors（推荐）
```

### 6.3 执行合并

```bash
llamafactory-cli export configs/lora_LlamaFactory_merge.yaml
```

合并完成后，`export_dir` 目录结构如下，可直接作为标准 HuggingFace 模型使用：

```
output/merged_model/
├── model-00001-of-00003.safetensors   # 合并后完整权重（分片）
├── model-00002-of-00003.safetensors
├── model-00003-of-00003.safetensors
├── model.safetensors.index.json
├── config.json
├── tokenizer.json
└── tokenizer_config.json
```

**验证合并后模型可正常推理：**

```bash
# 将推理配置中的 model_name_or_path 改为合并后目录，去掉 adapter_name_or_path
llamafactory-cli chat configs/lora_LlamaFactory_merged_infer.yaml
```

## 七、使用 vLLM 部署合并后的模型

完成权重合并后，`output/merged_model/` 目录中已经是标准的 HuggingFace 格式模型，可以直接交给 vLLM 进行高性能推理部署。

**vLLM 的核心优势**

| 特性 | 说明 |
|------|------|
| PagedAttention | 动态管理 KV Cache，大幅提升显存利用率 |
| Continuous Batching | 请求级动态批处理，吞吐量比静态批处理高 3~10× |
| OpenAI 兼容接口 | 无缝对接现有 OpenAI SDK 代码，无需改造客户端 |
| 多卡张量并行 | 单机多卡线性扩展，支持 70B+ 大模型 |
| 量化推理 | 支持 AWQ / GPTQ / FP8，显存压缩 50%~75% |

### 7.1 安装 vLLM

vLLM 需要与 CUDA 版本严格匹配，推荐使用 pip 安装预编译包：

> **⚠️ 强烈建议在虚拟环境（conda env / venv）下安装 vLLM。**  
> `pip install vllm` 会自动安装与其版本匹配的 `torch`、`transformers` 等依赖，若安装到全局环境或已有其他项目的环境中，可能造成依赖冲突，覆盖已有的 torch 版本，影响其他工程的正常运行。

```bash
# （可选）创建并激活独立的虚拟环境，避免依赖冲突
conda create -n vllm_env python=3.10 -y
conda activate vllm_env

# 安装 vLLM（会自动安装匹配版本的 torch、transformers 等依赖）
pip install vllm

# 验证安装，查看版本号
python -c "import vllm; print(vllm.__version__)"
```

> **注意**：  
> - `pip install vllm` 会自动拉取并安装对应版本的 **PyTorch**、**Transformers**，无需单独安装，但也正因如此，建议隔离环境操作。  
> - vLLM 与 torch 版本绑定较紧，若安装失败可参考官方文档指定版本：  
>   `pip install vllm==0.25.0`（当前最新稳定版，以 [GitHub Releases](https://github.com/vllm-project/vllm/releases) 为准）

### 7.2 常用启动参数说明

以下为 `vllm serve` 命令中最常用的参数，按功能分组说明：

#### 7.2.1 模型与精度

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `--model` | 必填 | 模型路径（本地目录或 HuggingFace ID）|
| `--dtype` | `auto` | 权重精度：`float16` / `bfloat16` / `auto` |
| `--trust-remote-code` | False | 允许执行模型仓库中的自定义代码；仅当模型含自定义建模代码时才需开启（见下方判断方法） |
| `--quantization` | None | 量化方式：`awq` / `gptq` / `fp8` |
| `--tokenizer` | 与 `--model` 相同 | 单独指定 tokenizer 的路径（本地目录或 HuggingFace ID）；LlamaFactory 导出合并模型后 `tokenizer_config.json` 中可能缺少 `chat_template` 字段，此时用 `--tokenizer` 指向原始基座模型目录，即可让 vLLM 从基座模型加载完整 tokenizer，而模型权重仍从合并目录读取 |
| `--chat-template` | None | 指定自定义 Jinja2 chat template 文件的路径（`.jinja` 文件）；优先级高于 `tokenizer_config.json` 中的 `chat_template` 字段；适用于 tokenizer 本身没有内置 chat template、或需要临时覆盖模板格式的场景 |

> **`--trust-remote-code` 是否需要开启？**
>
> LlamaFactory 合并后的模型**通常不需要**，是否需要取决于所使用的**基座模型**：
>
> - Llama 3 / Mistral / Gemma 等主流模型 → **不需要**，Transformers 已原生支持
> - Qwen2 及以上版本 → **不需要**，新版已原生支持
> - Qwen1.5 及更早版本 → **需要**，依赖自定义建模代码
> - ChatGLM3 / InternLM 老版本 → **需要**，包含自定义建模代码
>
> **快速判断方法**：查看合并后模型目录中的 `config.json`，若包含 `auto_map` 字段则说明依赖自定义代码，需要开启：
>
> ```json
> "auto_map": {
>   "AutoModelForCausalLM": "modeling_xxx.AutoModelForCausalLM"
> }
> ```
>
> 若无 `auto_map` 字段则不需要。也可先不加该参数启动，若报 `Please pass trust_remote_code=True` 错误再补充即可。

> **`--tokenizer` 与 `--chat-template` 的使用场景对比**

| 场景 | 推荐方案 |
| --- | --- |
| 合并模型缺少 `chat_template`（LlamaFactory 导出常见问题） | `--tokenizer /path/to/base/model`（复用基座模型完整 tokenizer） |
| 需要临时更换对话模板格式，不修改模型文件 | `--chat-template /path/to/template.jinja`（文件路径方式覆盖） |
| tokenizer 与模型权重来自不同目录 | `--tokenizer` 指向 tokenizer 目录，`--model` 指向权重目录 |

#### 7.2.2 显存与性能

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `--gpu-memory-utilization` | `0.9` | **全部显存**的占用上限（0~1），已包含模型权重；KV Cache 实际可用 = 总显存 × 该值 − 模型权重；值太小会触发 Preemption，值太大有 OOM 风险；显存充裕可调至 `0.95`，遇到 OOM 降至 `0.85` |
| `--max-model-len` | 模型配置 | 最大序列长度（输入+输出 token 总数）|
| `--max-num-seqs` | 自动推算 | 同时处理的最大请求数（并发上限），实际由引擎根据显存动态决定 |
| `--max-num-batched-tokens` | 自动推算 | 单次迭代处理的最大 token 总数（所有并发请求合计），调大可提升吞吐，需与 `--max-model-len` 配合 |
| `--enable-prefix-caching` | V0默认False，V1默认**True** | 开启自动前缀缓存（APC）：相同前缀（如 system prompt）的 KV Cache 自动复用，多轮对话 / 共享前缀场景可显著降低 TTFT；当前 stable 版默认使用 V1 引擎，APC 默认已开启 |
| `--enable-chunked-prefill` | V0默认False，V1**始终启用** | 开启分块 Prefill：将长 Prefill 切成小块与 Decode 交替执行，改善长输入请求对在途 Decode 请求的 ITL 影响；V1 引擎中此特性强制开启，`--no-enable-chunked-prefill` 无效 |

> **`--gpu-memory-utilization` 显存分配原理**  
> 该参数控制的是**总显存**的使用上限，模型权重已包含在内，vLLM 按以下顺序分配显存：  
> 1. 先将**模型权重**加载进显存（固定开销）  
> 2. 计算可用上限：`总显存 × gpu_memory_utilization`  
> 3. KV Cache 获得：`可用上限 − 模型权重`
>
> 显存充裕但 KV Cache 不够用 → 调高至 `0.95`；遇到 OOM → 降低至 `0.85`。

**示例（80 GB A100，加载 14B 模型约 28 GB，默认值 0.9）：**

| 项目 | 显存 |
| --- | --- |
| 总显存 | 80 GB |
| 可用上限（× 0.9） | 72 GB |
| 模型权重 | 28 GB |
| KV Cache 实际可用 | **44 GB** |
| 系统预留（剩余 10%） | 8 GB |

#### 7.2.3 多卡并行

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `--tensor-parallel-size` | `1` | 张量并行卡数，单机多卡时设为实际卡数 |

#### 7.2.4 服务接口

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `--host` | `localhost` | 监听地址，`0.0.0.0` 表示对外开放 |
| `--port` | `8000` | 监听端口 |
| `--api-key` | None | API 鉴权密钥（不设则无鉴权）|
| `--served-model-name` | None | 对外暴露的模型名称（影响 `/v1/models` 返回值）|

#### 7.2.5 推理模型（Reasoning Model）

对于带有思考链（Chain-of-Thought）的推理模型，需要添加以下参数，vLLM 才能正确识别并拆分思维链与最终回答：

| 参数 | 默认值 | 说明 |
|------|--------|------|
| `--reasoning-parser` | None | 指定推理模型的思维链解析器；vLLM 会自动将 `<think>...</think>` 中的内容拆分到响应的 `reasoning` 字段，`content` 字段只保留最终回答；**不加此参数时，思维链与回答混在 `content` 字段中**，需调用方自行解析 |

各推理模型系列对应的解析器值：

| 模型系列 | `--reasoning-parser` 值 |
|---------|------------------------|
| DeepSeek-R1 / R1-Distill 系列 | `deepseek_r1` |
| Qwen3 / QwQ 思考模式 | `qwen3` |
| Granite 3.2 | `granite` |

> **注意**：添加 `--reasoning-parser` 后，请求响应中思维链内容移至 `choices[0].message.reasoning` 字段，`content` 字段仅含最终答案。若调用方代码原先从 `content` 中截取 `</think>` 后的内容，需同步调整读取字段。

### 7.3 启动 vLLM 服务

将 `output/merged_model/` 中合并好的完整模型作为入参，启动 OpenAI 兼容的 HTTP 服务。

#### 7.3.1 单卡启动（7B 模型，显存 ≥ 16GB）

```bash
vllm serve output/merged_model \
  --dtype bfloat16 \
  --gpu-memory-utilization 0.90 \
  --max-model-len 4096 \
  --host 0.0.0.0 \
  --port 8000 \
  --served-model-name my-lora-model
```

#### 7.3.2 低延迟场景（限制并发，减少排队等待）

```bash
vllm serve output/merged_model \
  --dtype bfloat16 \
  --gpu-memory-utilization 0.85 \
  --max-model-len 4096 \
  --max-num-seqs 32 \
  --host 0.0.0.0 \
  --port 8000
```

> **为什么低延迟场景反而要调低 `gpu-memory-utilization` 到 0.85？**
>
> 原因在于 **Preemption（抢占）对延迟的伤害**：当 KV Cache 接近满时，vLLM 会触发 Preemption，把某条正在生成的请求"暂停并驱逐"，等显存释放后再重新调度（期间 KV Cache 需重算）。这会造成**不可预测的延迟毛刺**，对 P99 延迟影响极大。
>
> 逻辑链：`--max-num-seqs 32` 限制并发数少 → KV Cache 总需求小 → 不需要把显存利用率撑到 0.90 → 设为 0.85 多留余量 → Preemption 触发概率降低 → 延迟更稳定。
>
> 高吞吐场景（7.3.3）则反过来：接受一定延迟抖动，换取更大 KV Cache 容纳 256 个并发请求，因此将利用率推高到 0.95。

#### 7.3.3 高吞吐场景（允许更大批次，适合离线批量推理）

```bash
vllm serve output/merged_model \
  --dtype bfloat16 \
  --gpu-memory-utilization 0.95 \
  --max-model-len 4096 \
  --max-num-seqs 256 \
  --max-num-batched-tokens 8192 \
  --host 0.0.0.0 \
  --port 8000
```

> **服务启动成功标志**：终端输出 `INFO:     Application startup complete.` 且无报错，即可通过 HTTP 接口调用。

### 7.4 调用 API（curl 验证）

服务启动后，可用以下命令快速验证接口是否正常响应：

#### 7.4.1 查询可用模型列表

```bash
curl http://localhost:8000/v1/models
```

预期返回（`id` 即为 `--served-model-name` 指定的名称）：

```json
{
  "object": "list",
  "data": [
    {
      "id": "my-lora-model",
      "object": "model",
      "created": 1720000000,
      "owned_by": "vllm"
    }
  ]
}
```

#### 7.4.2 发起对话补全请求

```bash
curl http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "my-lora-model",
    "messages": [
      {"role": "user", "content": "请介绍一下大语言模型的工作原理"}
    ],
    "max_tokens": 512,
    "temperature": 0.7
  }'
```

**curl 参数说明：**

| 参数 | 全称 | 作用 |
|------|------|------|
| `-H` | `--header` | 添加 HTTP 请求头，告知服务器发送的数据格式；此处 `Content-Type: application/json` 表示请求体为 JSON，服务器据此用 JSON 解析器处理 |
| `-d` | `--data` | 设置 POST 请求体，即实际发送给服务器的 JSON 内容；加了 `-d` 后，curl 会自动将请求方式从 GET 改为 **POST** |

> 类比理解：`-H` 是快递外包装上的"内含易碎品"标签，`-d` 是包裹里装的实际物品。

#### 7.4.3 流式输出（打字机效果）

在请求体中加入 `"stream": true`，服务器将以 **SSE（Server-Sent Events）** 格式逐 token 实时返回，而不是等全部生成完再一次性返回：

```bash
curl http://localhost:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "my-lora-model",
    "messages": [
      {"role": "user", "content": "请介绍一下大语言模型的工作原理"}
    ],
    "max_tokens": 512,
    "temperature": 0.7,
    "stream": true
  }'
```

返回格式为多行 `data:` 事件流，每行携带一个 token 片段，最后以 `[DONE]` 结束：

```
data: {"choices": [{"delta": {"role": "assistant"}, "finish_reason": null}]}
data: {"choices": [{"delta": {"content": "大语"}, "finish_reason": null}]}
data: {"choices": [{"delta": {"content": "言模"}, "finish_reason": null}]}
data: {"choices": [{"delta": {"content": "型"}, "finish_reason": null}]}
...
data: [DONE]
```

**流式 vs 非流式对比：**

| 对比项 | `stream: false`（默认） | `stream: true` |
|--------|------------------------|----------------|
| 返回时机 | 全部生成完才返回 | 逐 token 边生成边返回 |
| 用户体验 | 等待后一次性显示 | 打字机效果，实时显示 |
| 适用场景 | 批处理、后端调用 | 对话界面、交互演示 |
| 协议格式 | 普通 HTTP JSON 响应 | SSE 流式响应（`text/event-stream`）|

#### 7.4.4 乱码问题排查（tokenizer 兼容性）

若 vLLM 服务能正常启动，但 API 返回内容出现 `Ċ`、`Ġ`、`ðŁĺĬ` 等异常字符，或中文完全丢失、只剩英文，**不一定是模型权重或合并步骤的问题**，更常见的原因是 **vLLM 虚拟环境中的 `transformers` 版本与模型目录中的 tokenizer 配置不兼容**，这一问题在 Llama、Qwen、DeepSeek 等主流系列模型上均可能出现。

---

**典型症状**

| 现象 | 说明 |
|------|------|
| 输出含 `Ċ`、`Ġ` | Byte-Level BPE 的原始 token 符号未被正确解码（`Ċ` ≈ 换行，`Ġ` ≈ 空格） |
| 中文输入，英文乱答 | tokenizer 编码中文失败，中文字符被映射为 `<unk>` 后丢弃 |
| LlamaFactory 推理正常，vLLM 乱码 | 两个环境的 transformers 版本不同，tokenizer 加载行为不一致 |

---

**根本原因**

```
vLLM ≥ 0.24.0  →  强制依赖 transformers v5，移除了 v4 支持
transformers v5  →  按 tokenizer_config.json 中的 tokenizer_class 字段加载 tokenizer
若 tokenizer_class 为旧版名称（如 LlamaTokenizerFast）  →  v5 下可能静默降级为慢速实现
慢速 tokenizer 依赖 tokenizer.model（SentencePiece 文件）  →  若该文件缺失，中文字符全部变为 <unk> 并被丢弃
→  vLLM 收到损坏的 token 序列  →  输出乱码
```

> **注意**：vLLM 0.24+ 强制依赖 transformers v5，**不能通过降级 transformers 来规避此问题**，必须修改模型目录的 tokenizer 配置。

---

**诊断方法**

在 **vLLM 虚拟环境**中对模型目录的 tokenizer 进行编码/解码测试，判断中文是否能正常往返：

- 若加载的 tokenizer 类型为 `LlamaTokenizer`（非 `Fast` 后缀）、token 数量异常少（仅 2~3 个）、且解码结果丢失中文，则说明 tokenizer 异常。
- 若类型为 `LlamaTokenizerFast` 或 `PreTrainedTokenizerFast`，且中英文均能正确还原，则 tokenizer 正常，乱码另有他因。

---

**修复思路**

打开模型目录下的 `tokenizer_config.json`，将 `tokenizer_class` 字段由旧版类名（如 `LlamaTokenizerFast`、`Qwen2Tokenizer` 等）改为通用的 `PreTrainedTokenizerFast`：

```json
// 修改前（旧版类名，v5 下可能无法正确加载）
"tokenizer_class": "LlamaTokenizerFast"

// 修改后（通用快速 tokenizer，直接从 tokenizer.json 加载，绕过类名兼容问题）
"tokenizer_class": "PreTrainedTokenizerFast"
```

修改后 transformers v5 会直接读取同目录下的 `tokenizer.json`（HuggingFace fast tokenizer 标准格式），不再依赖旧版类实现，中文编码恢复正常。

> **前提**：模型目录中必须存在完整的 `tokenizer.json` 文件（正常大小约 5~10 MB）。若该文件缺失或损坏，需先从原始基座模型目录复制或重新下载。

---

**修复后操作**

修改 `tokenizer_config.json` 后，**重启 vLLM 服务**，再次用 curl 发送含中文的请求，确认输出恢复正常。

---

**备选方案：覆盖 tokenizer 文件**

若不希望修改 `tokenizer_config.json`，也可将**原始基座模型**目录中的 tokenizer 相关文件整体复制并覆盖到模型目录：

```
tokenizer.json
tokenizer_config.json
special_tokens_map.json
```

原始基座模型的文件通常是与其发布时的 transformers 版本严格对齐的，覆盖后可同时解决类名不兼容和文件缺失两类问题。

---

**推理模型的额外注意事项**

对于带有思考链（Chain-of-Thought）的推理模型（如 DeepSeek-R1 系列、QwQ 系列），启动时建议添加对应的 `--reasoning-parser` 参数，让 vLLM 自动将思考内容与最终回答拆分到不同字段，避免 `<think>...</think>` 标签原文混入 `content` 字段：

| 模型系列 | 推荐参数 |
|---------|---------|
| DeepSeek-R1 / R1-Distill | `--reasoning-parser deepseek_r1` |
| Qwen3 思考模式 | `--reasoning-parser qwen3` |

```bash
# 示意：启动时附带 reasoning-parser
vllm serve output/merged_model \
  --reasoning-parser deepseek_r1 \
  --host 0.0.0.0 \
  --port 8000
```

### 7.5 Python 客户端调用示例

vLLM 提供 OpenAI 兼容接口，可直接使用 `openai` SDK 调用，无需修改任何客户端逻辑。

#### `stream` 参数说明

`stream` 是每次 API **请求体**中的字段（非服务启动参数），控制响应的返回方式：

| `stream` 值 | 默认 | 返回方式 | 适用场景 |
|-------------|------|----------|----------|
| `False` | ✅ | 等待生成完毕后一次性返回完整 JSON | 批量处理、后台任务、需要完整 usage 统计 |
| `True` | — | 逐 token 以 SSE（Server-Sent Events）格式实时推送 | 对话产品、用户交互、"打字机"效果 |

**非流式（默认）**

发请求 → 等待生成完毕 → 返回完整 JSON

**流式（stream=True）**

发请求 → `data: {"delta": "你"}` → `data: {"delta": "好"}` → `data: {"delta": "！"}` → `data: [DONE]`

> 流式模式下，最终 chunk 不含 `usage` 字段；若需要 token 用量统计，请使用非流式调用。

In [ ]:
"""
使用 openai SDK 调用本地 vLLM 服务。

功能：
    通过 OpenAI 兼容接口向 vLLM 发起 Chat Completion 请求，
    支持普通响应和流式响应两种模式。

依赖：
    pip install openai
"""

from openai import OpenAI  # 导入 OpenAI 官方 Python SDK（str: 客户端类）

# ── 基础配置 ──────────────────────────────────────────────
BASE_URL = "http://localhost:8000/v1"  # vLLM 服务地址（str: OpenAI 兼容端点）
API_KEY = "EMPTY"                      # 无鉴权时填任意非空字符串（str）
MODEL_NAME = "my-lora-model"           # 与 --served-model-name 保持一致（str）

# ── 初始化客户端 ──────────────────────────────────────────
client = OpenAI(
    base_url=BASE_URL,   # 覆盖默认的 OpenAI 官方地址，指向本地 vLLM（str）
    api_key=API_KEY,     # SDK 要求必填，本地服务填占位字符串即可（str）
)

# ── 构造对话消息列表 ───────────────────────────────────────
messages = [
    {
        "role": "system",                            # 系统提示，设定模型角色（str）
        "content": "你是一个专业的 AI 助手，擅长简洁清晰地回答问题。",
    },
    {
        "role": "user",                              # 用户输入（str）
        "content": "请用三句话解释什么是 LoRA 微调。",
    },
]

# ── 普通（非流式）调用 ─────────────────────────────────────
# temperature / top_p / max_tokens 是 OpenAI 标准参数，可直接传递（float / int）
# repetition_penalty 是 vLLM 扩展参数，openai SDK 不认识，需用 extra_body 透传给服务端
response = client.chat.completions.create(
    model=MODEL_NAME,    # 目标模型名称（str）
    messages=messages,   # 对话历史列表（list[dict]）
    max_tokens=512,      # 最多生成 512 个 token（int）
    temperature=0.7,     # 采样温度，0=贪心，越高越随机（float: 0~2）
    top_p=0.9,           # nucleus sampling 概率阈值（float: 0~1）
    extra_body={
        # vLLM 扩展字段，通过 extra_body 绕过 SDK 的参数校验直接传给服务端（dict）
        "repetition_penalty": 1.1,  # 重复惩罚系数，>1 抑制重复输出（float）
    },
)

# response 类型：openai.types.chat.ChatCompletion
# response.choices: list[Choice]，每个 Choice 包含一条候选回复
answer = response.choices[0].message.content  # 提取首条候选的文本内容（str）
print("=== 模型回复 ===")
print(answer)

# 打印 token 用量统计（prompt_tokens + completion_tokens = total_tokens）
usage = response.usage  # openai.types.CompletionUsage
print(f"\n输入 tokens：{usage.prompt_tokens}")      # int: 输入消耗的 token 数
print(f"输出 tokens：{usage.completion_tokens}")   # int: 生成消耗的 token 数
print(f"总计 tokens：{usage.total_tokens}")        # int: 两者之和

In [ ]:
"""
流式调用 vLLM 服务，逐 token 实时打印输出。

功能：
    使用 stream=True 参数开启流式模式，
    每收到一个 token chunk 就立即打印，实现"打字机"效果。
"""

# ── 流式调用 ──────────────────────────────────────────────
stream = client.chat.completions.create(
    model=MODEL_NAME,    # 目标模型名称（str）
    messages=messages,   # 对话历史列表（list[dict]）
    max_tokens=512,      # 最多生成 512 个 token（int）
    temperature=0.7,     # 采样温度（float）
    stream=True,         # 开启流式响应，返回生成器对象（bool）
)

# stream 类型：openai.Stream[ChatCompletionChunk]
# 每次迭代返回一个 ChatCompletionChunk，包含增量 delta 内容
print("=== 流式输出 ===")
for chunk in stream:  # chunk: openai.types.chat.ChatCompletionChunk
    delta = chunk.choices[0].delta          # 增量内容对象（ChoiceDelta）
    token_text = delta.content              # 当前 chunk 包含的文本片段（str | None）
    if token_text is not None:              # None 表示该 chunk 为结束标记
        print(token_text, end="", flush=True)  # 不换行立即刷新，模拟流式效果

print()  # 流式结束后换行，保持输出整洁

### 7.6 多卡部署与量化推理

#### 7.6.1 单机多卡（张量并行）

当模型权重超过单卡显存时（如 70B 模型需要 ~140GB），使用张量并行将模型切分到多张 GPU：

```bash
# 4 卡张量并行部署 70B 模型（每卡分担 ~35GB）
vllm serve output/merged_model \
  --tensor-parallel-size 4 \
  --dtype bfloat16 \
  --gpu-memory-utilization 0.90 \
  --max-model-len 8192 \
  --host 0.0.0.0 \
  --port 8000
```

> `--tensor-parallel-size` 必须能整除模型的 attention head 数，常见值为 2 / 4 / 8。

#### 7.6.2 AWQ 量化部署（显存减半）

若模型已经过 AWQ 量化（或使用 AutoAWQ 量化导出），可显著降低显存需求：

```bash
# 量化模型部署（AWQ INT4，7B 模型显存降至 ~4GB）
vllm serve output/merged_model-awq \
  --quantization awq \
  --dtype float16 \
  --gpu-memory-utilization 0.90 \
  --host 0.0.0.0 \
  --port 8000
```

#### 7.6.3 参数选择参考

| 场景 | `gpu_memory_utilization` | `max_num_seqs` | `tensor_parallel_size` |
|------|--------------------------|----------------|------------------------|
| 实时对话（低延迟优先）| 0.85 | 32 | 1 |
| 批量推理（高吞吐优先）| 0.95 | 256 | 1 |
| 大模型部署（显存不足）| 0.90 | 128 | 2 / 4 / 8 |
| 量化节省显存 | 0.90 | 128 | 1 |

### 7.7 性能评估指标

vLLM 自带两套完整的评估体系：**压测工具**（离线一次性跑出报告）和 **Prometheus 实时监控接口**（服务运行期间持续暴露指标）。

#### 7.7.1 核心指标说明

| 指标 | 全称 | 含义 | 影响因素 |
|------|------|------|----------|
| **TTFT** | Time To First Token | 从发请求到收到第一个 token 的时间（prefill 延迟）| 输入长度、`--max-num-batched-tokens`、并发数 |
| **TPOT** | Time Per Output Token | 每生成一个 token 的平均时间（decode 速度）| batch size、`--max-num-seqs` |
| **ITL** | Inter-Token Latency | 相邻两个输出 token 之间的间隔（与 TPOT 接近）| 同 TPOT |
| **E2E Latency** | End-to-End Latency | 完整请求从发出到接收完毕的总延迟 | TTFT + 输出长度 × TPOT |
| **Throughput** | — | 单位时间内处理的 token 数 / 请求数 | batch size、并发数、模型大小 |

**TTFT vs TPOT 的直觉理解**：

```
用户发请求
    │
    ▼  Prefill（处理全部输入 token，一次完成）
    │
    └──→ 输出第 1 个 token  ← TTFT（体感"响应快不快"）
              │
              ▼  Decode（每步生成 1 个 token，循环执行）
              │
              └──→ TPOT（体感"输出流不流畅"）
```

#### 7.7.2 压测工具（`vllm bench serve`）

服务启动后，使用官方内置压测命令模拟并发请求，自动统计并打印完整报告：

```bash
# 向运行中的 vLLM 服务发起 500 条请求，并发速率 10 req/s
# --dataset-name random 使用随机合成数据，无需准备真实数据集（默认值）
# --input-len / --output-len 控制随机数据集的输入/输出长度
vllm bench serve \
  --model my-lora-model \
  --base-url http://localhost:8000 \
  --num-prompts 500 \
  --request-rate 10 \
  --dataset-name random \
  --input-len 512 \
  --output-len 128 \
  --tokenizer output/merged_model
```

输出示例（关键字段）：

```
Successful requests: 500
Benchmark duration (s): 52.3
Total input tokens: 62500
Total generated tokens: 128000
Request throughput (req/s): 9.56
Output token throughput (tok/s): 2448.3
------------------ Latency ------------------
Mean TTFT (ms):   248.7
P50  TTFT (ms):   231.4
P99  TTFT (ms):   892.1
Mean TPOT (ms):    18.3
P50  TPOT (ms):    17.9
P99  TPOT (ms):    31.2
Mean E2E  (ms):  2871.5
```

#### 7.7.3 Prometheus 实时监控接口（`/metrics`）

vLLM 服务运行期间，通过 `/metrics` 端点持续暴露实时指标，可接入 Grafana 构建监控看板：

```bash
# 查看所有 Prometheus 格式指标
curl http://localhost:8000/metrics
```

核心指标名称：

| Prometheus 指标名 | 类型 | 说明 |
|-----------------|------|------|
| `vllm:time_to_first_token_seconds` | histogram | TTFT 分布（含 P50 / P95 / P99）|
| `vllm:inter_token_latency_seconds` | histogram | ITL / TPOT 分布，官方 Grafana Dashboard 使用此指标监控解码吞吐 |
| `vllm:request_time_per_output_token_seconds` | histogram | 每条请求维度的 TPOT 均值分布（与 ITL 互补，颗粒度不同）|
| `vllm:e2e_request_latency_seconds` | histogram | 端到端延迟分布（从收到请求到生成完毕）|
| `vllm:kv_cache_usage_perc` | gauge | KV Cache 实时占用率（0~1，1 表示 100%）|
| `vllm:num_requests_running` | gauge | 当前正在处理（decode）的请求数 |
| `vllm:num_requests_waiting` | gauge | 排队等待调度的请求数 |
| `vllm:request_prompt_tokens` | histogram | 每条请求的输入 token 数分布 |
| `vllm:request_generation_tokens` | histogram | 每条请求的输出 token 数分布 |

> **使用建议**：开发阶段用 `vllm bench serve` 快速得出基准数据；生产部署后接入 Prometheus + Grafana，通过实时看板监控 `vllm:kv_cache_usage_perc` 和 `vllm:num_requests_waiting`，及时发现性能瓶颈。

## 附：完整工作流总结

```
原始数据（JSON）
   │
   ▼ 二、注册到 data/dataset_info.json
   │
   ▼ 三、修改 configs/lora_LlamaFactory_train.yaml
   │
   ▼ llamafactory-cli train configs/lora_LlamaFactory_train.yaml
   │
   └─→ output/lora_sft/
           ├── adapter_model.safetensors   ← LoRA 权重
           └── training_loss.png
   │
   ├─▶ 四、推理（LoRA 动态加载）
   │    llamafactory-cli chat configs/lora_LlamaFactory_infer.yaml
   │
   ├─▶ 五、评估（两种方式，按需选择）
   │
   │    ① 内置基准评估（MMLU / C-Eval / CMMLU，多选题准确率）
   │    llamafactory-cli eval configs/lora_LlamaFactory_eval.yaml
   │    └─→ output/eval_results/
   │            ├── results.json    ← 各学科准确率明细
   │            └── summary.json   ← 整体平均准确率（average_acc）
   │
   │    ② 自定义数据集评估（任意数据集，BLEU / ROUGE）
   │    llamafactory-cli train configs/lora_LlamaFactory_predict.yaml
   │    └─→ output/predict_results/
   │            ├── generated_predictions.jsonl  ← 逐条生成结果
   │            └── predict_results.json         ← BLEU-4 / ROUGE-L 得分
   │
   ▼ 六、llamafactory-cli export configs/lora_LlamaFactory_merge.yaml
   │
   └─→ output/merged_model/   ← 标准 HuggingFace 完整模型
   │
   ▼ 七、vllm serve output/merged_model --host 0.0.0.0 --port 8000
   │
   └─→ OpenAI 兼容 HTTP 服务
           ├── GET  /v1/models              ← 查询可用模型
           ├── POST /v1/chat/completions    ← 对话补全接口
           └── GET  /metrics               ← Prometheus 实时指标
   │
   ▼ 七（可选）vllm bench serve --model my-lora-model --num-prompts 500
   │
   └─→ 压测报告：TTFT / TPOT / E2E Latency / Throughput
```

**下一步建议**
- 在 C-Eval / MMLU 等基准上对比微调前后 `average_acc`，量化微调收益
- 尝试 DPO / KTO 阶段对齐微调，提升模型的指令跟随能力
- 使用 AutoAWQ 对合并模型进行 INT4 量化，进一步压缩显存占用